---
# Document Structure Based Text Splitting in Langchain
---

### Introduction
- python code -> has class, function, for loop --> no. character word --> loose some info
- splitting will be happening on the basis of document specified structure


### Recursive Character Text Splitter
- based on language 
- this time we need to mention explicitiy about has python, html, json, java etc. 

# `Detailed Notes`

# Document Structure-Based Text Splitting in LangChain

> **Document structure-based text splitting divides a document according to its natural hierarchy—such as chapters, sections, headings, paragraphs, tables, or other logical units—instead of simply cutting the text after a fixed number of characters.**

The core idea is:

```text
Don't split only by SIZE.
Split according to the DOCUMENT'S MEANINGFUL STRUCTURE.
```

---

# 1. Why Do We Need Document Structure-Based Splitting?

Imagine a 200-page technical book:

```text
# Deep Learning

## Introduction

Deep learning is...

## Neural Networks

Neural networks are...

### Activation Functions

Activation functions...

### Loss Functions

Loss functions...

## CNN

Convolutional neural networks...
```

A fixed-size splitter might create:

```text
Chunk 1
──────────────
...Neural Networks...

### Activation Fu
```

Chunk 2:

```text
nctions...
```

The heading and its content can become separated.

A structure-based splitter tries to preserve:

```text
# Deep Learning
    │
    ├── ## Introduction
    │
    ├── ## Neural Networks
    │       ├── ### Activation Functions
    │       └── ### Loss Functions
    │
    └── ## CNN
```

This produces chunks that have more meaningful context.

---

# 2. What Is Document Structure?

A document can have different structural levels.

For example:

```text
Document
│
├── Chapter
│   │
│   ├── Section
│   │   │
│   │   ├── Subsection
│   │   │
│   │   └── Paragraph
│   │
│   └── Section
│
└── Chapter
```

Different document formats expose different structures.

### PDF / Book

```text
Book
 ↓
Chapter
 ↓
Section
 ↓
Subsection
 ↓
Paragraph
```

### Markdown

```text
# Heading
## Heading
### Heading
Paragraph
```

### HTML

```html
<h1>
<h2>
<h3>
<p>
```

### Code

```text
Class
 ↓
Method
 ↓
Statement
```

---

# 3. Basic Idea

Compare these two approaches.

### Length-Based

```text
Document
   ↓
Every 1000 characters
   ↓
Chunk 1
Chunk 2
Chunk 3
```

### Structure-Based

```text
Document
   ↓
Chapter / Section / Heading
   ↓
Meaningful Sections
   ↓
Chunks
```

So:

> **Length-based splitting asks "How much?" while structure-based splitting asks "Where does the meaning naturally change?"**

---

# 4. Example

Suppose your document contains:

```text
# Machine Learning

Machine learning allows computers to learn from data.

## Supervised Learning

Supervised learning uses labeled data.

### Classification

Classification predicts discrete categories.

### Regression

Regression predicts continuous values.

## Unsupervised Learning

Unsupervised learning works with unlabeled data.
```

A structure-aware splitter can preserve:

```text
Chunk 1
────────────
Machine Learning
Machine learning allows computers to learn from data.
```

```text
Chunk 2
────────────
Machine Learning
→ Supervised Learning

Supervised learning uses labeled data.
```

```text
Chunk 3
────────────
Machine Learning
→ Supervised Learning
→ Classification

Classification predicts discrete categories.
```

```text
Chunk 4
────────────
Machine Learning
→ Supervised Learning
→ Regression

Regression predicts continuous values.
```

Notice that the hierarchy is preserved.

---

# 5. Why Is Hierarchy Important for RAG?

Suppose the user asks:

> "What is classification?"

The retriever finds:

```text
Classification predicts discrete categories.
```

But this alone doesn't tell us that classification belongs to:

```text
Machine Learning
    ↓
Supervised Learning
    ↓
Classification
```

Metadata can preserve that relationship.

For example:

```python
{
    "chapter": "Machine Learning",
    "section": "Supervised Learning",
    "subsection": "Classification"
}
```

Now the LLM receives both:

```text
Content
+
Context
```

This can make the retrieved information easier to interpret.

---

# 6. Structure-Based Splitting in LangChain

LangChain provides different splitters depending on the document structure.

Common examples include:

```text
MarkdownHeaderTextSplitter
HTMLHeaderTextSplitter
RecursiveCharacterTextSplitter
Language-aware code splitting
```

For document structure, the most important concept is:

> **Use a splitter that understands the structure of the source document.**

---

# 7. Markdown Example

Suppose we have:

```markdown
# LangChain

LangChain is a framework for LLM applications.

## Models

Models interact with language models.

## Prompts

Prompts provide instructions to models.

## Agents

Agents can use tools to perform tasks.
```

We can split using Markdown headers.

```python
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

documents = splitter.split_text(markdown_text)
```

---

# 8. What Happens to the Metadata?

For example:

```text
## Agents

Agents can use tools to perform tasks.
```

could result conceptually in:

```python
Document(
    page_content="Agents can use tools to perform tasks.",
    metadata={
        "Header 1": "LangChain",
        "Header 2": "Agents"
    }
)
```

So the chunk knows where it came from.

---

# 9. Structure + Metadata

This is one of the most important concepts.

Instead of storing:

```text
Agents can use tools to perform tasks.
```

you can have:

```text
Content:
Agents can use tools to perform tasks.

Metadata:
Header 1 = LangChain
Header 2 = Agents
```

Therefore:

```text
Chunk
 ├── Content
 │
 └── Metadata
      ├── Parent section
      └── Current section
```

---

# 10. Document Hierarchy

Think of a technical document like a tree:

```text
Deep Learning
│
├── Neural Networks
│   │
│   ├── Perceptron
│   │
│   ├── Activation Functions
│   │
│   └── Backpropagation
│
├── CNN
│   │
│   ├── Convolution
│   ├── Pooling
│   └── CNN Architecture
│
└── RNN
    │
    ├── Sequence Modeling
    └── LSTM
```

A good structure-based splitting strategy attempts to retain this context.

---

# 11. Document Structure-Based Splitting vs Recursive Splitting

These are related but different.

### Document Structure-Based

Uses explicit document structure:

```text
Chapter
Section
Heading
Subheading
HTML element
Code construct
```

### Recursive Character Splitting

Uses a hierarchy of text separators:

```text
Paragraph
 ↓
Line
 ↓
Space
 ↓
Character
```

### Easy Way to Remember

```text
Structure-Based
→ Understand the document's hierarchy.

Recursive
→ Find a good boundary using separators.
```

---

# 12. Structure-Based Splitting vs Length-Based Splitting

| Feature              | Length-Based      | Structure-Based                 |
| -------------------- | ----------------- | ------------------------------- |
| Main focus           | Size              | Meaning/structure               |
| Uses                 | Characters/tokens | Headings/sections/etc.          |
| Context preservation | Limited           | Better                          |
| Predictable size     | Yes               | Not necessarily                 |
| Good for Markdown    | Sometimes         | Yes                             |
| Good for HTML        | Sometimes         | Yes                             |
| Good for books       | Basic             | Yes                             |
| Good for code        | Usually weak      | Better with code-aware splitter |

---

# 13. Important Limitation

Structure-based splitting does **not** mean:

> "Every section becomes one chunk."

Imagine:

```text
## Transformers

500 pages of content...
```

If you create one chunk:

```text
Chunk = 500 pages
```

that's obviously not useful.

Therefore, structure-based splitting often needs to be combined with size-based splitting.

---

# 14. Hybrid Approach

A strong production strategy is:

```text
Document
   ↓
Identify Structure
   ↓
Split by Sections
   ↓
Check Chunk Size
   ↓
Large Section?
   │
   ├── No → Keep
   │
   └── Yes
        ↓
   Further Split
```

For example:

```text
# Deep Learning
     ↓
## CNN
     ↓
10,000 characters
     ↓
Too large
     ↓
Recursive Character Splitter
     ↓
Chunk 1
Chunk 2
Chunk 3
```

This gives you:

> **Semantic structure + manageable chunk size**

---

# 15. Practical RAG Architecture

For a technical documentation RAG system:

```text
                 Documentation
                       ↓
              Document Structure
                       ↓
             Structure-Based Split
                       ↓
                 Logical Sections
                       ↓
          Size-Based Split if Needed
                       ↓
                    Chunks
                       ↓
                  Embeddings
                       ↓
                Vector Database
                       ↓
                  Retriever
                       ↓
              Relevant Context
                       ↓
                     LLM
                       ↓
                    Answer
```

---

# 16. Example: PDF Book

Suppose:

```text
Deep_Learning.pdf
```

contains:

```text
Chapter 1 → Introduction
Chapter 2 → Neural Networks
Chapter 3 → CNN
Chapter 4 → RNN
Chapter 5 → Transformers
```

Inside Chapter 3:

```text
CNN
├── Convolution
├── Filters
├── Pooling
└── Architecture
```

A good ingestion pipeline could preserve:

```text
chapter = CNN
section = Convolution
```

as metadata.

Then a retrieved chunk might look like:

```text
Metadata:
chapter: CNN
section: Convolution

Content:
A convolution operation applies a filter...
```

This is much better context for the LLM than an arbitrary text fragment.

---

# 17. Example: HTML Documentation

Suppose:

```html
<h1>LangChain</h1>

<h2>Models</h2>
<p>...</p>

<h2>Agents</h2>
<p>...</p>

<h2>Retrieval</h2>
<p>...</p>
```

A structure-aware pipeline can produce:

```text
LangChain → Models
LangChain → Agents
LangChain → Retrieval
```

with those headings available as metadata/context.

---

# 18. Example: Code

For a source-code RAG system:

```python
class UserService:

    def create_user(self):
        ...

    def delete_user(self):
        ...

class AuthService:

    def login(self):
        ...
```

You don't want arbitrary cuts like:

```text
class UserService:

    def create_use
```

Instead, a language-aware strategy can try to preserve:

```text
UserService
 ├── create_user()
 └── delete_user()

AuthService
 └── login()
```

This is especially useful for:

> **"Chat with my codebase"**

applications.

---

# 19. When Should You Use Document Structure-Based Splitting?

### Excellent use cases

* Technical documentation
* API documentation
* Markdown files
* HTML documentation
* Books
* Research papers
* Legal documents
* Product documentation
* GitHub repositories
* Source code

Especially when the document has a clear hierarchy.

---

# 20. When Is It Less Useful?

If the content has no meaningful structure:

```text
Random text
Raw logs
Unformatted text
Simple chat messages
```

then a character/token/recursive splitter may be more appropriate.

---

# 21. Real GenAI Project

## Project: Chat With Your Technical Notes

This is particularly useful for your own learning notes.

Suppose your folder contains:

```text
genai-notes/
│
├── langchain.md
├── rag.md
├── embeddings.md
├── agents.md
└── transformers.md
```

Each Markdown file contains:

```text
# Topic

## Definition

...

## Architecture

...

## Example

...

## Interview Questions

...
```

Pipeline:

```text
Markdown Files
      ↓
Document Loader
      ↓
Structure-Based Splitter
      ↓
Topic / Section / Subsection
      ↓
Optional Size-Based Split
      ↓
Embeddings
      ↓
Vector DB
      ↓
Retriever
      ↓
LLM
```

Then ask:

> "Explain LangChain Agents with an example."

The retriever can find:

```text
File: langchain.md
Topic: Agents
Section: Example
```

---

# 22. Interview Questions

## Beginner

### Q1. What is document structure-based text splitting?

**Answer:**

It divides a document according to its natural hierarchy, such as chapters, headings, sections, paragraphs, or code structures, instead of relying only on fixed character or token lengths.

---

### Q2. Why is document structure important in RAG?

**Answer:**

It helps preserve semantic relationships between a chunk and its surrounding context, which can improve the quality and interpretability of retrieved information.

---

### Q3. Give examples of document structure.

**Answer:**

Examples include:

```text
Chapter
Section
Heading
Subheading
Paragraph
Table
Code class
Code function
```

---

# 23. Intermediate Questions

### Q4. What is `MarkdownHeaderTextSplitter`?

**Answer:**

It is a LangChain text splitter that uses Markdown headers such as `#`, `##`, and `###` to split Markdown content while preserving header information as metadata.

---

### Q5. Why store headings in metadata?

**Answer:**

Headings provide context about where a chunk belongs in the document. This can help retrieval, filtering, debugging, and giving the LLM additional context.

---

### Q6. Can structure-based splitting alone handle a 100-page section?

**Answer:**

Not necessarily. A large section can still produce an oversized chunk. In practice, structure-based splitting is often followed by size-based splitting for sections that exceed the desired size.

---

# 24. Scenario-Based Questions

### Q7. You are building a RAG system for technical documentation. Which splitting approach would you choose?

**Answer:**

I'd first preserve the documentation hierarchy using a structure-aware splitter, such as a Markdown or HTML header splitter. Then I'd apply additional size-based splitting to sections that are too large.

---

### Q8. Your RAG retrieves correct content but the answer lacks context. What would you check?

I'd check whether:

```text
Parent headings
Section information
Metadata
Document hierarchy
```

are being preserved during ingestion.

---

### Q9. Would you use the same splitter for a PDF book and source code?

**Answer:**

No. The splitting strategy should match the source. A book may benefit from chapter/section-aware splitting, while source code benefits from language-aware splitting that understands classes, functions, and methods.

---

# 25. 30-Second Revision

> **Document structure-based splitting divides documents according to their natural hierarchy rather than arbitrary character counts.**

Remember:

```text
Document
   ↓
Chapter
   ↓
Section
   ↓
Subsection
   ↓
Content
```

Examples:

```text
Markdown → Headers
HTML → Headers / Elements
Books → Chapters / Sections
Code → Classes / Functions
```

### Main Benefit

```text
Structure
   ↓
Meaningful chunks
   ↓
Better context
   ↓
Better RAG retrieval
```

---

# 26. 2-Minute Revision

## Document Structure-Based Text Splitting

### Definition

> Splitting a document according to its natural logical structure instead of only using a fixed size.

### Examples

```text
PDF / Book
→ Chapter → Section → Subsection

Markdown
→ # → ## → ###

HTML
→ h1 → h2 → h3

Code
→ Class → Function → Method
```

### LangChain Example

```python
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers
)

chunks = splitter.split_text(markdown_text)
```

### Production Strategy

```text
Document
   ↓
Structure-Based Split
   ↓
Preserve Hierarchy
   ↓
Check Size
   ↓
Further Split Large Sections
   ↓
Embeddings
   ↓
Vector DB
```

### Key Interview Point

> **Structure-based splitting preserves the semantic hierarchy of documents, while size-based splitting controls the amount of text in each chunk. A strong RAG pipeline often combines both: preserve structure first, then split oversized sections into manageable chunks.**

### Memory Trick

```text
Length-Based
→ "How BIG?"

Structure-Based
→ "WHERE does the meaning change?"

Hybrid
→ "Preserve meaning + control size."
```
